In [262]:
!pip install torch
!pip install torch_geometric
!pip install rdkit
!pip install torcheval
!pip install scipy
!pip install scikit-learn
!pip install matplotlib
!git clone https://github.com/olsson-group/sml_project1.git

fatal: destination path 'sml_project1' already exists and is not an empty directory.


In [263]:
from abc import abstractmethod

import warnings
warnings.filterwarnings("ignore")

import torch
import torch_geometric as geom
from torch_geometric.datasets import MoleculeNet
import matplotlib.pyplot as plt

from sml_project1 import metrics as sml_metrics
from sml_project1 import data as sml_data

## Build model classes

### Use the below structure to build wrapper modules for the model backbones

E.g for a given backbone architecture (DeepSet, GCN, GAT), wrap it in a regression/classifcation module which provides training and eval functionality. See classes for backbone architectures below.

In [ ]:
class MolecularGraphNet(torch.nn.Module):
    # This is the parent class for both the regression and classification task.
    # NOTE: You should not need to modify this class

    def __init__(self, embedding, net):
        super().__init__()
        self.embedding = embedding
        self.net = net

    def training_step(self, loss):
        # print(loss)
        loss.backward()
        self.optimizer.step()
        self.optimizer.zero_grad()

    def _setup_optimizer(self):
        self.optimizer = torch.optim.Adam(self.parameters(), lr=0.001)

    @abstractmethod
    def forward(self, batch):
        # Implement in children
        raise NotImplementedError

    @abstractmethod
    def loss(self, y, y_hat):
        # Implement in children
        raise NotImplementedError


class MolecularGraphClassification(MolecularGraphNet):
    # Class for molecular graph classification
    # TODO: Finish implementing the class

    def __init__(self, embedding, net):
        super().__init__(embedding, net)
        self._setup_optimizer()

    def forward(self, batch):
        x = batch.x.to(torch.int32)  # prevent a type mismatch

        x = self.embedding(x)  
        # print(x.shape)
        x = self.net(x, batch.edge_index, batch.batch)
        # print(x.shape)
        x = geom.nn.global_mean_pool(x, batch.batch)
        # print(x.shape)

        return x
        # TODO: Setup the forward pass of the model for a classification task, this includes embedding the input data, applying the graph network, pooling the results, and predicting an output.
        # HINT: Pooling can be done using: x = geom.nn.global_mean_pool(batch.x, batch.batch) where batch.x denotes the node features, and batch.batch denotes which graph each node belongs to
        # raise NotImplementedError

    def loss(self, y, y_hat):
        loss_fun = torch.nn.BCELoss()

        return  loss_fun(y_hat, y)

        # # TODO: Setup the appropriate loss for training a classification model (see assignment pdf)
        # raise NotImplementedError
    

    def evaluate(self, y, y_hat):
        roc_auc = sml_metrics.get_auc_roc(y, y_hat)
        return roc_auc
    

class MolecularGraphRegression(MolecularGraphNet):
    # Class for molecular graph regression
    # TODO: Finish implementing the class

    def __init__(self, embedding, net):
        super().__init__(embedding, net)
        self._setup_optimizer()

        self.output_layer = torch.nn.Linear(net.out_channels, 1)
        self.output_activation = torch.nn.ReLU()

    def forward(self, batch):
        x = batch.x.to(torch.int32)  # prevent a type mismatch

        A = geom.utils.to_dense_adj(batch.edge_index, batch=batch.batch)
        print(A)
        print(batch.edge_index)
        print(batch.batch)
        print(x.shape)
        x = self.embedding(x)
        print(x.shape)
        raise NotImplementedError
        x = self.net(x, batch.edge_index, batch.batch)
        # print(x.shape)
        # print(x)
        x = geom.nn.global_mean_pool(x, batch.batch)
        # print(x.shape)

        x = self.output_layer(x)    
        x = self.output_activation(x)
        return x
        
        # # TODO: Setup the forward pass of the model for a classification task, this includes embedding the input data, applying the graph network, pooling the results, and predicting an output.
        # raise NotImplementedError

    def loss(self, y, y_hat):
        loss_fun = torch.nn.MSELoss()
        return loss_fun(y_hat, y)
    
        # # TODO: Setup the appropriate loss for training a regression model (see assignment pdf)
        # raise NotImplementedError

    def evaluate(self, y, y_hat):
        pcc, _ = sml_metrics.get_pearson_corr(y, y_hat)
        rmse = sml_metrics.get_rmse(y, y_hat)
        return pcc, rmse

### Use the below structure to build embeddings for molecular graph node features
Before inputting node features to the classification/regression model backbones, you need to featurise them with an embedding. Use the below structure to build your node embeddings  

In [265]:
class MolecularNodeEmbedding(torch.nn.Module):
    # TODO: Implement an embedding for molecular node features

    def __init__(self, n_embeddings: list, min_vals: list, max_vals: list, emb_dim: int):
        super().__init__()

        self.min_vals = min_vals
        self.max_vals = max_vals
        
        self.embeddings = torch.nn.ModuleList()

        for i in range(len(n_embeddings)):
            embedding = torch.nn.Embedding(n_embeddings[i], emb_dim)
            self.embeddings.append(embedding)

        # print(len(self.embeddings))
        # # TODO: Construct an embedding per node feature (see: https://docs.pytorch.org/docs/2.14/generated/torch.nn.Embedding.html) 
        # raise NotImplementedError 
    
    def forward(self, x):
        # print(x.shape)

        # print(x[:2])

        x = torch.sub(x, torch.tensor(embedding_statistics["min_vals"]).unsqueeze(0))

        # print(x.shape)
        # print(x[:2])

        embedded_features = []
        for i in range(x.shape[1]):
            embedded_feature = self.embeddings[i](x[:, i])
            embedded_features.append(embedded_feature)

        x = torch.cat(embedded_features, dim=1)
        # print(x.shape)


        return x
        # # NOTE: Each node feature in a molecular graph needs to be embedded independently!
        
        # # TODO: Ensure correct input values of x have the correct range before embedding. 
        # # (HINT: after embedding, the features can be concatenated to a joint feature vector)  
        # raise NotImplementedError
    

def get_embedding_statistics(dataset):
    """
    For each input feature, find nr of embeddings and min/max values across dataset.

    TODO: Use this function to construct an embedding per node feature!
    """

    x_feats = torch.concat([item.x for item in dataset], dim=0)

    min_vals = x_feats.min(dim=0).values
    max_vals = x_feats.max(dim=0).values
    n_embeddings = max_vals - min_vals + 1

    stats = {
        "min_vals": min_vals.tolist(),  # minimum value of each node feature across the dataset
        "max_vals": max_vals.tolist(),  # maximum value of each node feature across the dataset
        "n_embeddings": n_embeddings.tolist()  # nr embeddings per node feature
    }
    return stats    

### Regression/classification model backbones

Implement classes below to compare DeepSet, GCN and GAT model architectures. 

In [266]:
class DeepSet(torch.nn.Module):
  # TODO: Implement DeepSet model.
  def __init__(self, in_channels, hidden_channels, out_channels):
    super().__init__()

    self.in_channels = in_channels
    self.hidden_channels = hidden_channels
    self.out_channels = out_channels

    self.model = torch.nn.Sequential(
        torch.nn.Linear(in_channels, hidden_channels[0]),
        torch.nn.ReLU(),
        torch.nn.Linear(hidden_channels[0], out_channels),
        torch.nn.ReLU()
    )

    # raise NotImplementedError

  def forward(self, x, edge_index, batch):
    return self.model(x)

    # raise NotImplementedError

In [267]:
class GCN(torch.nn.Module):
  # TODO: Implement GCN model.
  def __init__(self, in_channels, hidden_channels, out_channels):
    super().__init__()
    raise NotImplementedError

  def forward(self, x, edge_index, batch):
    raise NotImplementedError


In [268]:
class GAT(torch.nn.Module):
  # TODO: Implement GAT model.
  def __init__(self, in_channels, hidden_channels, out_channels):
    super().__init__()
    raise NotImplementedError


  def forward(self, x, edge_index, batch):
    raise NotImplementedError


## Regression task

### Load ESOL dataset

In [269]:
# REGRESSION TASK: DATA

# Download ESOL data, filter small molecules and split into training and validation dataset
esol = sml_data.filter_dataset(MoleculeNet(root="./storage/", name="ESOL"))

node_features = 9  # There are 9 features in the ESOL dataset

# TODO: Calculate nr of embeddings per atom feature and standardise the node feature ranges 
# (HINT: use the function *get_embedding_statistics* defined above)
embedding_statistics = get_embedding_statistics(esol)

print(embedding_statistics["min_vals"])
print(embedding_statistics["max_vals"])
print(embedding_statistics["n_embeddings"])
# x_feats = torch.concat([item.x for item in esol], dim=0)
# print(len(esol))
# for i, item in enumerate(esol):
#     esol[i].x = item.x - torch.tensor(embedding_statistics["min_vals"]).unsqueeze(0)  # standardise the node feature ranges



# Split data in train/val/test
train_dataset, val_dataset, test_dataset = sml_data.split_dataset(esol, frac_train=0.8, seed=0, split="random")  # TODO: complete this (HINT: check sml_project1/data.py for data splitting options)

[6, 0, 1, 4, 0, 0, 2, 0, 0]
[53, 0, 4, 6, 3, 0, 4, 1, 1]
[48, 1, 4, 3, 4, 1, 3, 2, 2]


### Define models for ESOL regression 

In [270]:
# REGRESSION TASK: MODELS
emb_dim = 5

in_features = 9 * emb_dim  # TODO: Calculate backbone input dimension from node_features (see above)
hidden_features = [64]
out_features = 3

# Embedding
# TODO: implement an embedding for each of your models (HINT: see helper class MolecularNodeEmbedding above)
deepset_embedding = MolecularNodeEmbedding(
    embedding_statistics["n_embeddings"],
    embedding_statistics["min_vals"],
    embedding_statistics["max_vals"],
    emb_dim=emb_dim,
)
deepset_regression = MolecularGraphRegression(
    deepset_embedding,
    DeepSet(in_features, hidden_features, out_features),
)

# gcn_embedding = ...
# gcn_regression = MolecularGraphRegression(
#     gcn_embedding,
#     GCN(in_features, hidden_features, out_features),
# )

# gat_embedding = ...
# gat_regression = MolecularGraphRegression(
#     gat_embedding,
#     GAT(in_features, hidden_features, out_features),
# )

### Train and evaluate models for ESOL regression

In [271]:
# REGRESSION TASK: TRAINING

train_batch_size = 2
val_batch_size = 10
n_epochs = 10

train_dataloader = geom.loader.DataLoader(train_dataset, batch_size=train_batch_size, shuffle=True)
val_dataloader = geom.loader.DataLoader(val_dataset, batch_size=val_batch_size)

# NOTE: This is an example for training of one model. 
# TODO: Extend code below to evaluate each architecture (see models defined above).
model = deepset_regression

# EXAMPLE of training script
for epoch in range(n_epochs):
  # train
  model.train()
  for batch in train_dataloader:
    y_hat = model.forward(batch)
    train_loss = model.loss(batch.y, y_hat)
    model.training_step(train_loss)

  # eval
  model.eval()
  with torch.inference_mode():
    y_hats = []  # NOTE: Ensure you average/calculate metrics based on entire validation batch 
    ys = []
    for val_batch in val_dataloader:
      y_hat = model.forward(val_batch)
      val_loss = model.loss(val_batch.y, y_hat)
      # print(val_batch.y.shape)
      # print(y_hat.shape)
      val_metrics = model.evaluate(val_batch.y, y_hat)

      ys.append(val_batch.y)
      y_hats.append(y_hat)

    ys, y_hats = torch.cat(ys), torch.cat(y_hats)
    val_rho, val_rmse = model.evaluate(ys, y_hats)

AttributeError: module 'torch' has no attribute 'matrix'

In [ ]:
# TODO: Present results for the regression task and compare model types (see instructions in the assignment pdf).
# This can be in the form of tables or graphs or however you see fit, ensure your comparison is comparable between architectures and model runs 



## Classification task

### Load BBBP dataset

In [ ]:
# CLASSIFICATION TASK: DATA

bbbp = sml_data.filter_dataset(MoleculeNet(root="./storage/", name="BBBP"))
node_features = 9  # There are 9 features in the BBBP dataset

# TODO: Re-calculate nr of embeddings per atom feature and standardise the node feature ranges to fit new dataset
embedding_statistics = get_embedding_statistics(bbbp)

print(embedding_statistics["min_vals"])
print(embedding_statistics["max_vals"])
print(embedding_statistics["n_embeddings"])

# Split data in train/val/test
train_dataset, val_dataset, test_dataset = sml_data.split_dataset(bbbp, frac_train=0.8, seed=..., split=...)  # TODO: complete this (HINT: check sml_project1/data.py for data splitting options)

[1, 0, 0, 4, 0, 0, 1, 0, 0]
[53, 2, 4, 7, 3, 1, 4, 1, 1]
[53, 3, 5, 4, 4, 2, 4, 2, 2]


AssertionError: split must be either 'random' or 'scaffold' random split provides a completely random split of all molecules in the datasret, while scaffold split provides a split based on the scaffold of the molecules in the dataset. Scaffold splitting is useful when you want to split the dataset based on the chemical similarity of the molecules in the dataset, but note that splitting by scaffold validation data may or may not include 'easier' molecules that the training data.

### Define models for BBBP classification

In [ ]:
# CLASSIFICATION TASK: MODELS

in_features = ...  # TODO: Calculate backbone input dimension from node_features (see above)
hidden_features = ...
out_features = ...

# Embedding
# TODO: implement an embedding for each of your models (HINT: see helper class MolecularNodeEmbedding above)
deepset_embedding = ...
deepset_classification = MolecularGraphClassification(
    deepset_embedding,
    DeepSet(in_features, hidden_features, 1),
)

gcn_embedding = ...
gcn_classification = MolecularGraphClassification(
    gcn_embedding,
    GCN(in_features, hidden_features, 1),
)

gat_embedding = ...
gat_classification = MolecularGraphClassification(
    gat_embedding,
    GAT(in_features, hidden_features, 1),
)

### Train and evaluate models for BBBP classification

In [ ]:
# CLASSIFICATION TASK: TRAINING
train_batch_size = ...
val_batch_size = ...
n_epochs = ...

train_dataloader = geom.loader.DataLoader(train_dataset, batch_size=train_batch_size, shuffle=True)
val_dataloader = geom.loader.DataLoader(val_dataset, batch_size=val_batch_size)

model = ...

# TODO: Again, train 3 models of each type and evaluate their loss and metrics

In [ ]:
# Present results for the classification task in way that is structurally comparable to the regression task